# Query UAV Localization (Flight 01)

End-to-end demo: run Steps 1–4 on a UAV image and Step 5b (KD-Tree retrieval + plurality vote) against the satellite database built by [`01_build_satellite_database.ipynb`](01_build_satellite_database.ipynb).

**Demo target**: `UAV-VisLoc/01/drone/01_0022.JPG` against `satellite01.tif` (9774 × 26762).

The UAV image is preprocessed to 500 × 500 and goes through the SAME `segment_batch` function used by every satellite patch in notebook 1, so segmentation quality is fair on both branches.

Output: a 3-panel figure showing the full satellite map with red ✕ (prediction) + magenta ★ (ground truth) connected by a yellow line annotated with the error distance, plus a zoom panel and the UAV view.

## 1. Clone repo & dependencies (skip if already done)

If you ran notebook 1 in the same session, you can jump to step 4.

In [ ]:
import os

REPO_URL = 'https://github.com/kagtgi/LocalizationUAV.git'
REPO_DIR = 'LocalizationUAV'

if not os.path.exists(REPO_DIR) and not os.path.exists('localization'):
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !pip install --quiet -r requirements.txt

## 2. Kaggle setup (only if data is not already on disk)

In [ ]:
if not os.path.exists('UAV-VisLoc/01/satellite01.tif'):
    # Setup Kaggle API
    !mkdir /.kaggle
    !mv kaggle.json /.kaggle
    !mv /.kaggle /root/
    !chmod 600 ~/.kaggle/kaggle.json

    !kaggle datasets download building-segment
    !kaggle datasets download hailong1610/uav-visloc-dataset
    !unzip -q building-segment.zip
    !unzip -q uav-visloc-dataset.zip
    print('UAV-VisLoc dataset downloaded and extracted')

## 2b. Download the Mask R-CNN checkpoint

`best_model.pth` (~170 MB) is hosted on Google Drive:

https://drive.google.com/file/d/1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB/view

The cell below downloads it on first run and skips on subsequent runs.

In [ ]:
MODEL_GDRIVE_ID = '1jlRfOXYU18DcOjWEwNBet1b7FHCIClcB'
MODEL_LOCAL_PATH = 'best_model.pth'

if not os.path.exists(MODEL_LOCAL_PATH):
    !pip install --quiet gdown
    !gdown 'https://drive.google.com/uc?id={MODEL_GDRIVE_ID}' -O {MODEL_LOCAL_PATH}
else:
    print(f'{MODEL_LOCAL_PATH} already present; skipping download.')

print('best_model.pth size:', os.path.getsize(MODEL_LOCAL_PATH) // (1024 * 1024), 'MB')

## 3. Imports + config

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / 'UAV-VisLoc'
FLIGHT_ID = '01'
MODEL_PATH = REPO_ROOT / 'best_model.pth'
UAV_IMAGE_NAME = '01_0022.JPG'

OUTPUT_DIR = REPO_ROOT / 'outputs' / FLIGHT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_NPZ_PATH = OUTPUT_DIR / f'satellite{FLIGHT_ID}_kdtree.npz'

TOP_K = 5  # paper §4.5

print('Repo  :', REPO_ROOT)
print('DB    :', DB_NPZ_PATH)
print('Model :', MODEL_PATH)
assert DB_NPZ_PATH.exists(), 'Run notebook 01 first to build the satellite database.'

In [ ]:
import os

import numpy as np
import torch
from PIL import Image

from localization import load_model, process_uav, SatelliteDatabase, query_uav
from localization.database.builder import extract_patch_descriptors
from localization.io.bounds import (
    load_satellite_bounds,
    latlon_to_pixel,
    pixel_to_latlon,
    pixel_offset_to_meters,
)
from localization.io.dataset import VisLocFlight, load_flight_metadata, get_image_pose
from localization.matching.visualize import (
    draw_gt_and_prediction,
    render_localization_result,
)

flight = VisLocFlight(flight_id=FLIGHT_ID, root=DATA_ROOT)
uav_image_path = flight.drone_image_path(UAV_IMAGE_NAME)
print('UAV image :', uav_image_path)
print('Sat TIF   :', flight.satellite_tif)
assert uav_image_path.exists(), f'Missing UAV image: {uav_image_path}'

## 4. Load Mask R-CNN + satellite database

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_model(
    model_path=str(MODEL_PATH),
    device=device,
    num_classes=2,
    pretrained=False,
).to(device).eval()
print('Device:', device)

db = SatelliteDatabase.load(str(DB_NPZ_PATH))
print(db)

## 5. Step 1 — Preprocess the UAV image

Yaw alignment from compass heading, altitude scale, central crop, resize to 500 × 500.

In [ ]:
img_raw, img_uav_500, meta = process_uav(
    img_path=str(uav_image_path),
    csv_path=str(flight.metadata_csv),
)
print('Pose meta:', meta)
img_uav_500

In [ ]:
preprocessed_path = OUTPUT_DIR / f'{Path(UAV_IMAGE_NAME).stem}_preprocessed.jpg'
img_uav_500.save(preprocessed_path, quality=95)
print('Saved preprocessed UAV image:', preprocessed_path)

## 6. Steps 2–4 — Segment, triangulate, MFCA → 5-D descriptors

Runs the same `segment_batch`/CDT/MFCA pipeline as each satellite patch (single image, `batch_size=1`).

In [ ]:
uav_descriptors, uav_centroids = extract_patch_descriptors(
    patch_image=img_uav_500,
    model=model,
    device=device,
    score_threshold=0.5,
)
print(f'UAV descriptors: {uav_descriptors.shape} ; centroids: {uav_centroids.shape}')
assert uav_descriptors.shape[0] > 0, 'No buildings segmented; try another query image.'
print('First 3 descriptors:')
print(uav_descriptors[:3])

## 7. Step 5b — KD-Tree query + plurality vote

In [ ]:
result = query_uav(uav_descriptors, db, k=TOP_K)
assert result is not None
print('Predicted patch  :', result.patch_id)
top_votes = dict(sorted(result.all_votes.items(), key=lambda kv: -kv[1])[:3])
print('Top-3 patch votes:', top_votes)
print('Margin           :', result.margin)
print('Predicted pixel  :', result.pixel_xy)

## 8. Convert prediction → lat/lon, compare against GT

In [ ]:
bounds = load_satellite_bounds(
    satellite_filename=os.path.basename(str(flight.satellite_tif)),
    csv_path=str(flight.bounds_csv),
)

with Image.open(flight.satellite_tif) as sat:
    sat_w, sat_h = sat.size

pred_lat, pred_lon = (None, None)
if bounds is not None:
    pred_lat, pred_lon = pixel_to_latlon(result.pixel_xy[0], result.pixel_xy[1], bounds, sat_w, sat_h)
    print(f'Predicted GPS : lat={pred_lat:.6f}, lon={pred_lon:.6f}')

metadata_df = load_flight_metadata(flight.metadata_csv)
row = get_image_pose(metadata_df, UAV_IMAGE_NAME)
gt_pixel = None
error_distance_m = None
if row is not None and bounds is not None:
    gt_lat, gt_lon = float(row['lat']), float(row['lon'])
    gt_pixel = latlon_to_pixel(gt_lat, gt_lon, bounds, sat_w, sat_h)
    print(f'GT GPS        : lat={gt_lat:.6f}, lon={gt_lon:.6f}')
    print(f'GT pixel      : {gt_pixel}')
    dx_px = result.pixel_xy[0] - gt_pixel[0]
    dy_px = result.pixel_xy[1] - gt_pixel[1]
    offset = pixel_offset_to_meters(dx_px, dy_px, bounds, sat_w, sat_h)
    error_distance_m = float(offset['distance_m'])
    print(f'Offset        : dx={offset["dx_m"]:.1f} m, dy={offset["dy_m"]:.1f} m, distance={error_distance_m:.1f} m')

## 9. Visualize GT and predicted positions on the satellite image

Three panels:

- **Left**: full `satellite01.tif` with the predicted position (red ✕), the ground-truth position (magenta ★), and a yellow connector annotated with the error in meters.
- **Center**: zoom around the prediction, with the same markers.
- **Right**: the UAV view used as the query.

In [ ]:
match_fig_path = OUTPUT_DIR / f'match_{Path(UAV_IMAGE_NAME).stem}.png'
title = f'Localization result: {UAV_IMAGE_NAME}  ->  {result.patch_id}'
if error_distance_m is not None:
    title += f'   (error: {error_distance_m:.1f} m)'

fig = render_localization_result(
    uav_image_path=str(uav_image_path),
    satellite_image_path=str(flight.satellite_tif),
    predicted_pixel_xy=result.pixel_xy,
    gt_pixel_xy=gt_pixel,
    error_distance_m=error_distance_m,
    zoom_radius_px=1500,
    title=title,
    output_path=str(match_fig_path),
)
fig

In [ ]:
print('3-panel figure saved to:', match_fig_path)

### Optional: save a standalone overlay of just the satellite + both markers

Useful for embedding in a report or paper figure.

In [ ]:
import cv2

sat_overlay = draw_gt_and_prediction(
    satellite_image_path=str(flight.satellite_tif),
    predicted_pixel_xy=result.pixel_xy,
    gt_pixel_xy=gt_pixel,
    error_distance_m=error_distance_m,
)
overlay_path = OUTPUT_DIR / f'overlay_{Path(UAV_IMAGE_NAME).stem}.jpg'
cv2.imwrite(str(overlay_path), cv2.cvtColor(sat_overlay, cv2.COLOR_RGB2BGR))
print('Satellite overlay saved to:', overlay_path)